In [1]:
!pip -q install transformers datasets torch scikit-learn evaluate accelerate wandb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2026.1.0 requires fsspec==2026.1.0, but you have fsspec 2025.10.0 which is incompatible.

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from datasets import load_dataset
import csv
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

raw_datasets = load_dataset(
    "csv",
    data_files={
        "train": "../../Data/LIAR/train.tsv",
        "validation": "../../Data/LIAR/valid.tsv",
        "test": "../../Data/LIAR/test.tsv",
    },
    delimiter="\t",
    column_names=col_names,
    quoting=csv.QUOTE_NONE,
)

# Mapping des 6 classes
label_mapping = {
    'pants-fire': 0, 'false': 1, 'barely-true': 2, 
    'half-true': 3, 'mostly-true': 4, 'true': 5
}
target_names = ['Pants-Fire', 'False', 'Barely-True', 'Half-True', 'Mostly-True', 'True']

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

raw_datasets = raw_datasets.map(map_labels)

Generating train split: 10269 examples [00:00, 135400.51 examples/s]
Generating validation split: 1284 examples [00:00, 108536.78 examples/s]
Generating test split: 1283 examples [00:00, 78358.82 examples/s]
Map: 100%|██████████| 1283/1283 [00:00<00:00, 10877.32 examples/s]


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import datasets
import wandb
import os
from transformers import BertConfig

my_secret_key = os.environ.get("WANDB")

wandb.login(key=my_secret_key)

wandb.init(project="liar-bert-fine-grained")

statement_train, y_train = raw_datasets["train"]["statement"], raw_datasets["train"]["label"]
statement_val, y_val = raw_datasets["validation"]["statement"], raw_datasets["validation"]["label"]
statement_test, y_test = raw_datasets["test"]["statement"], raw_datasets["test"]["label"]

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)


class StatementDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    max_length=self.max_len,
                    padding='max_length',
                    truncation=True,
                    return_attention_mask=True,
                    return_tensors='pt',
                )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_dataset = StatementDataset(statement_train, y_train, tokenizer)
val_dataset = StatementDataset(statement_val, y_val, tokenizer)
test_dataset = StatementDataset(statement_test, y_test, tokenizer)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 5. Chargement du modèle
# num_labels doit correspondre au nombre de classes dans vos données (ex: 2 pour binaire)
num_labels = len(set(y_train)) 

# 1. Charger la configuration par défaut
config = BertConfig.from_pretrained(model_name, num_labels=num_labels)

# 2. Augmenter le Dropout (Par défaut c'est 0.1)
# hidden_dropout_prob : Dropout sur les couches cachées (embeddings, encodeurs)
config.hidden_dropout_prob = 0.3  
# attention_probs_dropout_prob : Dropout sur les mécanismes d'attention
config.attention_probs_dropout_prob = 0.3 

# 3. Charger le modèle avec cette configuration modifiée
model = BertForSequenceClassification.from_pretrained(model_name, config=config)

# 6. Configuration de l'entraînement
training_args = TrainingArguments(
    output_dir='./results',          # Dossier de sortie
    num_train_epochs=4,              # Nombre d'époques
    per_device_train_batch_size=16,  # Taille du batch d'entraînement
    per_device_eval_batch_size=32,   # Taille du batch d'évaluation
    warmup_steps=500,                # Steps de chauffe pour le learning rate
    weight_decay=0.01,               # Régularisation
    logging_dir='./logs',
    report_to="wandb",          # Active l'intégration Weights & Biases
    
    logging_steps=10,           # Enregistre la TRAINING loss tous les 10 batches
    
    eval_strategy="steps",      # Permet d'évaluer PENDANT l'époque (et non juste à la fin)
    eval_steps=50,              # Lance la validation tous les 50 batches (ajuster selon vitesse)
    
    save_strategy="steps",
    save_steps=500,
    
    load_best_model_at_end=True,
    metric_for_best_model="f1"  # Optionnel : choisit le meilleur modèle basé sur le F1 score plutôt que la loss
)

# 7. Création du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 8. Lancer l'entraînement
trainer.train()

# 9. Evaluation finale sur le test set
results = trainer.evaluate(test_dataset)
print("Résultats sur le test set :", results)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Faire des prédictions sur le jeu de test
print("Génération des prédictions...")
predictions_output = trainer.predict(test_dataset)

# 2. Récupérer les labels prédits (argmax sur les logits) et les vrais labels
preds = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

# 3. Récupérer les noms des classes (labels) depuis le dataset original
# Le dataset LIAR contient 6 classes : pants-fire, false, barely-true, half-true, mostly-true, true
target_names = raw_datasets["train"].features["label"].names

# 4. Afficher le rapport de classification complet
print("\n--- RÉSULTATS DÉTAILLÉS PAR LABEL ---")
report = classification_report(true_labels, preds, target_names=target_names)
print(report)

# --- BONUS : Matrice de Confusion ---
# Si vous voulez visualiser les erreurs entre classes
cm = confusion_matrix(true_labels, preds)

# Affichage simple dans la console
print("\n--- MATRICE DE CONFUSION ---")
print(cm)

In [ ]:
# (Assurez-vous que wandb.init() est toujours actif, donc AVANT wandb.finish())

# Créer un tableau WandB pour le rapport par classe
wandb.log({"conf_mat" : wandb.plot.confusion_matrix(probs=None,
                        y_true=true_labels, preds=preds,
                        class_names=target_names)})

# Si vous voulez le rapport textuel dans les logs
wandb.log({"classification_report": wandb.Html(f"<pre>{report}</pre>")})

wandb.finish()